# 1. Retriver 평가

### 1) 경로 설정

In [1]:
import sys
from pathlib import Path

# ✅ 노트북 위치가 어디든 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while (ROOT / "src").exists() is False and ROOT.parent != ROOT:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

ROOT = c:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project


### 2) 평가 데이터셋 로드

In [78]:
import json

DATA_PATH = ROOT / "evaluation" / "test_questions.json"  # 본인 파일명으로
with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

len(dataset), dataset[0].keys()

(30, dict_keys(['id', 'document', 'question', 'answer', 'source_location']))

### 3) 평가용 retriever 함수

In [79]:
from src.retriever import rerank_docs

def retrieve_docs(question: str, k: int):
    # candidate_k는 rerank에 넣을 후보 수 (top_n보다 넉넉히)
    candidate_k = max(60, k * 3)
    return rerank_docs(
        question,
        top_n=k,                 # ✅ K개 반환
        candidate_k=candidate_k, # ✅ 후보는 넉넉히
        docid_scope_top_n=None,  # ✅ 평가에서는 필터 OFF
    )

### 4) Recall, MRR 계산 함수

In [80]:
import os, re
from typing import Any, List, Dict, Optional, Tuple

def _norm_fname(s: str) -> str:
    """basename + lower + 공백 정리"""
    if not s:
        return ""
    s = os.path.basename(str(s)).strip()
    s = re.sub(r"\s+", " ", s)
    return s.lower()

def _doc_fname(doc) -> str:
    md = getattr(doc, "metadata", None) or {}
    fn = md.get("file_name") or md.get("filename") or md.get("source") or md.get("path") or ""
    return _norm_fname(fn)

def _first_rank(docs: List[Any], gold_document: str, k: int) -> Optional[int]:
    """상위 k개에서 정답 문서가 최초 등장하는 rank(1-based). 없으면 None"""
    gold = _norm_fname(gold_document)
    for i, d in enumerate((docs or [])[:k], 1):
        if _doc_fname(d) == gold:
            return i
    return None

def recall_mrr_at_k(docs: List[Any], gold_document: str, k: int) -> Tuple[float, float]:
    r = _first_rank(docs, gold_document, k)
    if r is None:
        return 0.0, 0.0
    return 1.0, 1.0 / r

In [81]:
from typing import Callable

def evaluate_retrieval(
    dataset: List[Dict[str, Any]],
    retrieve_fn: Callable[[str, int], List[Any]],
    k_list: List[int] = [1, 3, 5, 10],
    debug_n: int = 0,   # 처음 N개만 검색 결과 출력 (0이면 미출력)
):
    n = len(dataset)
    recall_sum = {k: 0.0 for k in k_list}
    mrr_sum = {k: 0.0 for k in k_list}

    for idx, item in enumerate(dataset, 1):
        q = item["question"]
        gold_doc = item["document"]

        # ✅ 가장 큰 K로 한 번만 가져오고, 슬라이싱으로 재사용 (속도 개선)
        max_k = max(k_list)
        docs = retrieve_fn(q, max_k)

        for k in k_list:
            r, m = recall_mrr_at_k(docs, gold_doc, k)
            recall_sum[k] += r
            mrr_sum[k] += m

        if debug_n and idx <= debug_n:
            print(f"\n[DEBUG Q{idx:02d}] {q}")
            print("GOLD_DOC:", gold_doc)
            print("TOP docs:")
            for j, d in enumerate(docs[:min(5, len(docs))], 1):
                print(f"  {j}. {_doc_fname(d)}")

    results = {
        "n": n,
        "Recall": {k: recall_sum[k] / n for k in k_list},
        "MRR": {k: mrr_sum[k] / n for k in k_list},
    }
    return results

### 5) 기본 평가

In [82]:
import pandas as pd

k_list = [1, 3, 5, 10, 20]  # 보고 싶은 K로 조정 (단, rerank_docs top_n이 이만큼 나와야 함)

results = evaluate_retrieval(
    dataset=dataset,
    retrieve_fn=retrieve_docs,
    k_list=k_list,
    debug_n=2,  # 처음 2개만 디버그 출력
)

df = pd.DataFrame({
    "K": list(results["Recall"].keys()),
    "Recall@K": list(results["Recall"].values()),
    "MRR@K": list(results["MRR"].values()),
})
df

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG Q01] 이 사업의 예산(배정예산)은 총 얼마이며, 어떤 부서가 주관하나요?
GOLD_DOC: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
TOP docs:
  1. 국가철도공단_철도인프라 디지털트윈 정보화전략계획(isp) 수립 용역(변.hwp
  2. 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp
  3. 한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp
  4. 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp
  5. 수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ismp 수립 입.hwp


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG Q02] 사업 범위 중 '재난위험지역 경보발령 범위 확대'를 위해 사용되는 무선 통신 방식은?
GOLD_DOC: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
TOP docs:
  1. 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp
  2. 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp
  3. 한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
  5. 서울특별시 여성가족재단_(재공고, 협상) 서울 디지털성범죄 안심지원센.hwp


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embedding

,K,Recall@K,MRR@K
0,1,0.066667,0.066667
1,3,0.066667,0.066667
2,5,0.133333,0.083333
3,10,0.200000,0.090000
4,20,0.266667,0.093627


# 2. BM25 / Dense(MMR) / Hybrid(RRF) / Rerank(FlashRank) 비교 평가

### 1) 경로 설정

In [83]:
import sys, importlib
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.retriever as retriever
importlib.reload(retriever)
print("retriever file:", retriever.__file__)

retriever file: c:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project\src\retriever.py


### 2) 공통 유틸 + Recall/MRR 평가 함수

In [84]:
import os, re
from typing import Any, List, Dict, Optional, Tuple, Callable
import pandas as pd

def norm_fname(s: str) -> str:
    if not s:
        return ""
    s = os.path.basename(str(s)).strip()
    s = re.sub(r"\s+", " ", s)
    return s.lower()

def doc_fname(doc) -> str:
    md = getattr(doc, "metadata", None) or {}
    fn = md.get("file_name") or md.get("filename") or md.get("source") or md.get("path") or ""
    return norm_fname(fn)

def first_rank(docs: List[Any], gold_document: str, k: int) -> Optional[int]:
    gold = norm_fname(gold_document)
    for i, d in enumerate((docs or [])[:k], 1):
        if doc_fname(d) == gold:
            return i
    return None

def recall_mrr_at_k(docs: List[Any], gold_document: str, k: int) -> Tuple[float, float]:
    r = first_rank(docs, gold_document, k)
    if r is None:
        return 0.0, 0.0
    return 1.0, 1.0 / r

def evaluate_retrieval(
    dataset: List[Dict[str, Any]],
    retrieve_maxk_fn: Callable[[str, int], List[Any]],
    k_list: List[int] = [1, 3, 5, 10, 20],
    debug_n: int = 0,
    label: str = "",
):
    n = len(dataset)
    max_k = max(k_list)
    recall_sum = {k: 0.0 for k in k_list}
    mrr_sum = {k: 0.0 for k in k_list}

    for idx, item in enumerate(dataset, 1):
        q = item["question"]
        gold_doc = item["document"]

        docs = retrieve_maxk_fn(q, max_k) or []

        for k in k_list:
            r, m = recall_mrr_at_k(docs, gold_doc, k)
            recall_sum[k] += r
            mrr_sum[k] += m

        if debug_n and idx <= debug_n:
            print(f"\n[DEBUG:{label} Q{idx:02d}] {q}")
            print("GOLD_DOC:", gold_doc)
            print("TOP5:")
            for j, d in enumerate(docs[:5], 1):
                print(f"  {j}. {doc_fname(d)}")

    return {
        "n": n,
        "Recall": {k: recall_sum[k] / n for k in k_list},
        "MRR": {k: mrr_sum[k] / n for k in k_list},
    }

### 3) 단계별 retriever 함수

In [85]:
### BM25 ###
def retrieve_bm25(question: str, k: int):
    r = retriever.get_bm25_retriever()
    docs = r.invoke(question) or []
    return docs[:k]


### DENSE ###
def retrieve_dense(question: str, k: int):
    r = retriever.get_dense_retriever()
    docs = r.invoke(question) or []
    return docs[:k]


### Hybird ###
import inspect

_hsig = inspect.signature(retriever.get_hybrid_docs)
_hparams = _hsig.parameters

def retrieve_hybrid(question: str, k: int):
    if "k" in _hparams:
        docs = retriever.get_hybrid_docs(question, k=k) or []
        return docs[:k]
    docs = retriever.get_hybrid_docs(question) or []
    return docs[:k]


### Rerank ###
_rsig = inspect.signature(retriever.rerank_docs)
_rparams = _rsig.parameters

def retrieve_rerank(question: str, k: int):
    if "top_n" in _rparams:
        candidate_k = max(60, k * 3) if "candidate_k" in _rparams else None
        kwargs = {"top_n": k}
        if candidate_k is not None:
            kwargs["candidate_k"] = candidate_k
        if "docid_scope_top_n" in _rparams:
            kwargs["docid_scope_top_n"] = None  # 평가용: 필터 OFF
        return retriever.rerank_docs(question, **kwargs) or []
    docs = retriever.rerank_docs(question) or []
    return docs[:k]

### 4) 단계별 평가 및 비교

In [86]:
k_list = [1, 3, 5, 10, 20]   # 원하는 K

runs = [
    ("BM25",   retrieve_bm25),
    ("Dense",  retrieve_dense),
    ("Hybrid", retrieve_hybrid),
    ("Rerank", retrieve_rerank),
]

rows = []
for name, fn in runs:
    res = evaluate_retrieval(dataset, fn, k_list=k_list, debug_n=0, label=name)
    for k in k_list:
        rows.append({
            "Stage": name,
            "K": k,
            "Recall@K": res["Recall"][k],
            "MRR@K": res["MRR"][k],
        })

df = pd.DataFrame(rows).sort_values(["K", "Stage"])
df

[BM25] loaded docs=19387


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embedding

,Stage,K,Recall@K,MRR@K
0,BM25,1,0.100000,0.100000
5,Dense,1,0.133333,0.133333
10,Hybrid,1,0.133333,0.133333
15,Rerank,1,0.066667,0.066667
1,BM25,3,0.133333,0.116667
6,Dense,3,0.166667,0.150000
11,Hybrid,3,0.200000,0.166667
16,Rerank,3,0.066667,0.066667
2,BM25,5,0.166667,0.125000
7,Dense,5,0.166667,0.150000


In [90]:
pivot_recall = df.pivot(index="K", columns="Stage", values="Recall@K")
pivot_mrr    = df.pivot(index="K", columns="Stage", values="MRR@K")

**Recall**

In [94]:
pivot_recall

Stage,BM25,Dense,Hybrid,Rerank
K,,,,
1,0.100000,0.133333,0.133333,0.066667
3,0.133333,0.166667,0.200000,0.066667
5,0.166667,0.166667,0.200000,0.133333
10,0.166667,0.166667,0.233333,0.200000
20,0.200000,0.233333,0.233333,0.266667


**MRR**

In [92]:
pivot_mrr

Stage,BM25,Dense,Hybrid,Rerank
K,,,,
1,0.100000,0.133333,0.133333,0.066667
3,0.116667,0.150000,0.166667,0.066667
5,0.125000,0.150000,0.166667,0.083333
10,0.125000,0.150000,0.172222,0.090000
20,0.128030,0.153704,0.172222,0.093627


# 3. 평가 데이터셋 변경

### 1) 경로 설정

In [60]:
import sys, importlib
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.retriever as retriever
importlib.reload(retriever)
print("retriever file:", retriever.__file__)

retriever file: c:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project\src\retriever.py


### 2) 수정된 평가 데이터셋

In [96]:
import json

DATA_PATH = ROOT / "evaluation" / "test_questions_2.json"  # 본인 파일명으로
with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

len(dataset), dataset[0].keys()

(30, dict_keys(['id', 'question', 'answer', 'evidence']))

### 3) 평가 함수

In [97]:
import os, re, unicodedata
from typing import Any, List, Dict, Optional, Tuple, Callable
import pandas as pd

def norm_doc_id(s: str) -> str:
    if not s:
        return ""
    s = unicodedata.normalize("NFC", str(s))
    s = os.path.basename(s).strip()
    s = re.sub(r"\.[A-Za-z0-9]+$", "", s)  # ✅ 확장자 제거
    s = re.sub(r"\s+", " ", s)
    return s.lower()

def gold_doc_ids_from_item(item):
    evs = item.get("evidence") or []
    out = []
    for ev in evs:
        f = ev.get("source_file")
        if f:
            did = norm_doc_id(f)
            if did not in out:
                out.append(did)
    return out

def doc_id_from_doc(doc):
    md = getattr(doc, "metadata", None) or {}
    fn = md.get("file_name") or md.get("filename") or md.get("source") or md.get("path") or ""
    return norm_doc_id(fn)


def first_rank_multi(docs: List[Any], gold_files: List[str], k: int) -> Optional[int]:
    """
    상위 k개 docs에서 gold_files 중 하나라도 최초 등장하는 rank(1-based)
    """
    gold_set = set(gold_files)
    for i, d in enumerate((docs or [])[:k], 1):
        if doc_id_from_doc(d) in gold_set:
            return i
    return None

def recall_mrr_at_k_multi(docs: List[Any], gold_files: List[str], k: int) -> Tuple[float, float]:
    r = first_rank_multi(docs, gold_files, k)
    if r is None:
        return 0.0, 0.0
    return 1.0, 1.0 / r

def evaluate_retrieval(
    dataset: List[Dict[str, Any]],
    retrieve_fn: Callable[[str, int], List[Any]],
    k_list: List[int] = [1, 3, 5, 10, 20],
    debug_n: int = 0,
    label: str = "",
):
    n = len(dataset)
    max_k = max(k_list)

    recall_sum = {k: 0.0 for k in k_list}
    mrr_sum = {k: 0.0 for k in k_list}

    for idx, item in enumerate(dataset, 1):
        q = item["question"]
        gold_files = gold_doc_ids_from_item(item)  # ✅ 이름 맞추기

        docs = retrieve_fn(q, max_k) or []

        for k in k_list:
            r, m = recall_mrr_at_k_multi(docs, gold_files, k)
            recall_sum[k] += r
            mrr_sum[k] += m

        if debug_n and idx <= debug_n:
            print(f"\n[DEBUG:{label} Q{idx:02d}] {q}")
            print("GOLD_DOC_IDS:")
            for gf in gold_files:
                print(" -", gf)
            print("TOP5:")
            for j, d in enumerate(docs[:5], 1):
                print(f"  {j}. {doc_id_from_doc(d)}")  # ✅ 평가 기준과 동일하게 출력

    return {
        "n": n,
        "Recall": {k: recall_sum[k] / n for k in k_list},
        "MRR": {k: mrr_sum[k] / n for k in k_list},
    }

### 4) 단계별 retriever 함수

In [ ]:
### BM25 ###
def retrieve_bm25(question: str, k: int):
    r = retriever.get_bm25_retriever()
    docs = r.invoke(question) or []
    return docs[:k]


### Dense ###
def retrieve_dense(question: str, k: int):
    r = retriever.get_dense_retriever()
    docs = r.invoke(question) or []
    return docs[:k]


### Hybird ###
import inspect

_hsig = inspect.signature(retriever.get_hybrid_docs)
_hparams = _hsig.parameters

def retrieve_hybrid(question: str, k: int):
    candidate_k = max(60, k * 3)

    # 신버전(get_hybrid_docs(question, k=...))이면 candidate_k를 넣기
    if "k" in _hparams:
        docs = retriever.get_hybrid_docs(question, k=candidate_k) or []
        return docs[:k]

    # 구버전이면 그냥 받아서 slice (구버전은 내부 k=60 고정일 확률 높음)
    docs = retriever.get_hybrid_docs(question) or []
    return docs[:k]


### Rerank ###
_rsig = inspect.signature(retriever.rerank_docs)
_rparams = _rsig.parameters

def retrieve_rerank(question: str, k: int):
    if "top_n" in _rparams:
        candidate_k = max(60, k * 3) if "candidate_k" in _rparams else None
        kwargs = {"top_n": k}
        if candidate_k is not None:
            kwargs["candidate_k"] = candidate_k
        if "docid_scope_top_n" in _rparams:
            kwargs["docid_scope_top_n"] = None  # 평가용: 필터 OFF
        return retriever.rerank_docs(question, **kwargs) or []
    docs = retriever.rerank_docs(question) or []
    return docs[:k]

### 5) 단계별 평가 및 비교

In [73]:
k_list = [1, 3, 5, 10, 20]
runs = [
    ("BM25",   retrieve_bm25),
    ("Dense",  retrieve_dense),
    ("Hybrid", retrieve_hybrid),
    ("Rerank", retrieve_rerank),
]

rows = []
for name, fn in runs:
    res = evaluate_retrieval(dataset, fn, k_list=k_list, debug_n=3, label=name)
    for k in k_list:
        rows.append({"Stage": name, "K": k, "Recall@K": res["Recall"][k], "MRR@K": res["MRR"][k]})

df = pd.DataFrame(rows).sort_values(["K", "Stage"])
df


[DEBUG:BM25 Q01] 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 부서가 어디인지도 확인 부탁드립니다.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축
  4. 서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육
  5. 국방과학연구소_대용량 자료전송시스템 고도화

[DEBUG:BM25 Q02] 이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무엇인가요?
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 경기도 평택시_2024년도 평택시 버스정보시스템(bis) 구축사업
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  5. 대검찰청_아태 사이버범죄 역량강화 허브(apc-hub) 홈페이지 및 온라인 교

[DEBUG:BM25 Q03] 봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기준을 알려주세요.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축
  3. 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG:Dense Q01] 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 부서가 어디인지도 확인 부탁드립니다.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 국가철도공단_철도인프라 디지털트윈 정보화전략계획(isp) 수립 용역(변
  3. 광주과학기술원_실시간통합연구비관리시스템(rcms) 연계 모듈 변경 사업
  4. 한국사학진흥재단_대학재정정보시스템(기본재산 및 기채 사후관리) 고
  5. 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 

[DEBUG:Dense Q02] 이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무엇인가요?
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. (사）한국대학스포츠협의회_kusf 체육특기자 경기기록 관리시스템 개발
  4. koica 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 
  5. 국가철도공단_철도인프라 디지털트윈 정보화전략계획(isp) 수립 용역(변


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG:Dense Q03] 봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기준을 알려주세요.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 한국수자원조사기술원_수문자료정보관리시스템(hdims) 재구축 용역(3단계
  3. 서울특별시 여성가족재단_(재공고, 협상) 서울 디지털성범죄 안심지원센
  4. 인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 erp시스템 구축 
  5. 축산물품질평가원_축산물이력관리시스템 개선(정보화 사업)


...


[DEBUG:Rerank Q01] 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 부서가 어디인지도 확인 부탁드립니다.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  5. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG:Rerank Q02] 이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무엇인가요?
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  5. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)

[DEBUG:Rerank Q03] 봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기준을 알려주세요.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  5. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embedding

,Stage,K,Recall@K,MRR@K
0,BM25,1,0.500000,0.500000
5,Dense,1,0.533333,0.533333
10,Hybrid,1,0.566667,0.566667
15,Rerank,1,0.433333,0.433333
1,BM25,3,0.566667,0.533333
6,Dense,3,0.733333,0.622222
11,Hybrid,3,0.700000,0.627778
16,Rerank,3,0.633333,0.527778
2,BM25,5,0.600000,0.541667
7,Dense,5,0.733333,0.622222


In [74]:
pivot_recall = df.pivot(index="K", columns="Stage", values="Recall@K")
pivot_mrr    = df.pivot(index="K", columns="Stage", values="MRR@K")

**Recall**

In [75]:
pivot_recall

Stage,BM25,Dense,Hybrid,Rerank
K,,,,
1,0.500000,0.533333,0.566667,0.433333
3,0.566667,0.733333,0.700000,0.633333
5,0.600000,0.733333,0.866667,0.766667
10,0.700000,0.833333,0.900000,0.800000
20,0.766667,0.900000,0.933333,0.933333


**MRR**

In [76]:
pivot_mrr

Stage,BM25,Dense,Hybrid,Rerank
K,,,,
1,0.500000,0.533333,0.566667,0.433333
3,0.533333,0.622222,0.627778,0.527778
5,0.541667,0.622222,0.664444,0.557778
10,0.553704,0.636706,0.668611,0.561111
20,0.557934,0.640750,0.670278,0.570438


# 4. 평가 데이터셋 활용하여 generator 평가

### 1) 환경 설정

In [10]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# 1) 노트북 위치가 어디든 프로젝트 루트 탐색 (src 폴더 기준)
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

assert (ROOT / "src").exists(), f"프로젝트 루트를 못 찾았습니다. 시작 cwd={Path.cwd()}"

# 2) 상대경로/로더 안정화: cwd를 루트로 이동
os.chdir(ROOT)

# 3) import 안정화: 루트를 sys.path 최상단에 1회만 추가
root_str = str(ROOT)
if root_str not in sys.path:
    sys.path.insert(0, root_str)

# 4) .env 로드
load_dotenv(ROOT / ".env")

print("cwd:", Path.cwd())
print("ROOT:", ROOT)
print("ROOT in sys.path:", root_str in sys.path)
print("OPENAI_API_KEY exists?:", bool(os.getenv("OPENAI_API_KEY")))

cwd: C:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project
ROOT: C:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project
ROOT in sys.path: True
OPENAI_API_KEY exists?: True


In [12]:
EVAL_PY = ROOT / "evaluation" / "evaluate.py"
DATASET = ROOT / "evaluation" / "test_questions_2.json"

assert EVAL_PY.exists(), f"evaluate.py 없음: {EVAL_PY}"
assert DATASET.exists(), f"test_questions_2.json 없음: {DATASET}"

print("EVAL_PY:", EVAL_PY)
print("DATASET:", DATASET)

EVAL_PY: C:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project\evaluation\evaluate.py
DATASET: C:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project\evaluation\test_questions_2.json


### 2) generator 체인 생성

In [13]:
from src.generator import get_rag_chain

chain = get_rag_chain()

# (선택) 스모크 테스트
print(chain.invoke("남서울대 학사행정 암호화 사업 추진 배경에 언급된 기존 시스템의 주요 보안 취약점은 무엇인가요?")[:300])
# print(ask(chain, "남서울대 학사행정 암호화 사업 추진 배경에 언급된 기존 시스템의 주요 보안 취약점은 무엇인가요?"))

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


ANSWER: 기존에 파일 암호화가 도입되어 있지 않아 원본 첨부파일 유출 시 개인정보가 그대로 노출될 위험이 있으며, 웹방화벽·서버 IP 정책 등 외부 접근 통제는 존재하나 보안사고 특성상 100% 안전하지 않음

SOURCES:
- file_name=남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp | doc_id=남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사 | chunk=4 | source_type=hwp_chunk_md
- file_name=남서울대학교_[혁신-국고] 남서울


### 3) 평가 실행 (soft_score)기반

In [14]:
from evaluation.evaluate import run_evaluation

results = run_evaluation(
    chain=chain,
    dataset_path=str(DATASET),
    threshold=0.35,
    use_langfuse=False,
    pred_preview_chars=300,
    retriever=None,   # LLM judge까지 쓰려면 아래 5) 참고
)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


RAG 시스템 평가 시작...



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


================= DEBUG Q01 =================
Q: 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 부서가 어디인지도 확인 부탁드립니다.
GOLD: 900,000,000원(VAT 포함)이며, 봉화군 안전건설과에서 주관합니다.

--- PRED_CORE(300) ---
사업금액: 900,000,000원(부가세 포함). 주관부서: 근거 문서에서 확인할 수 없습니다

Q01: [X]


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


...

================= DEBUG Q28 =================
Q: 꿀 등급 판정을 수행하는 현장 인력을 위해 이번에 새롭게 고도화되는 모바일 기능은 무엇인가요?
GOLD: 태블릿 PC 기반의 모바일 등급판정 시스템 고도화입니다.

--- PRED_CORE(300) ---
품질평가사가 시료채취 후 드럼봉인 시 봉인기에 각인된 드럼관리번호를 관리하기 위한 기능 고도화

Q28: [X]


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


================= DEBUG Q29 =================
Q: 꿀 품질평가 사업 보안 지침상 '경미한 위반 3건' 이상 발생 시 어떤 등급의 위약금을 물게 되나요?
GOLD: D급(200만 원 이하 위약금)입니다.

--- PRED_CORE(300) ---
근거 문서에서 확인할 수 없습니다

Q29: [X]


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


================= DEBUG Q30 =================
Q: 꿀 품질관리 시스템 누출금지 대상정보 중에서 2번 항목에 해당하는 구체적인 정보는 무엇인가요?
GOLD: 누출금지 대상정보 2번 항목입니다.(세부 정보시스템 구성현황 및 정보통신망 구성도)

--- PRED_CORE(300) ---
근거 문서에서 확인할 수 없습니다

Q30: [X]

평가 완료: 5 / 30


### 4) LLM-as-a-judge 평가

In [15]:
from src.generator import get_db_cached
from evaluation.evaluate import run_evaluation

db = get_db_cached()
retriever = db.as_retriever(search_kwargs={"k": 6})

results_llm_as_a_judge = run_evaluation(
    chain=chain,
    dataset_path=str(DATASET),
    threshold=0.35,
    retriever=retriever,  # ✅ llm_judge_score 활성화
)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


RAG 시스템 평가 시작...



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


================= DEBUG Q01 =================
Q: 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 부서가 어디인지도 확인 부탁드립니다.
GOLD: 900,000,000원(VAT 포함)이며, 봉화군 안전건설과에서 주관합니다.

--- PRED_CORE(300) ---
사업금액: 900,000,000원(부가세 포함)
주관 부서: 근거 문서에서 확인할 수 없습니다

Q01: [X]


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


...

================= DEBUG Q28 =================
Q: 꿀 등급 판정을 수행하는 현장 인력을 위해 이번에 새롭게 고도화되는 모바일 기능은 무엇인가요?
GOLD: 태블릿 PC 기반의 모바일 등급판정 시스템 고도화입니다.

--- PRED_CORE(300) ---
품질평가사가 시료채취 후 드럼봉인 시 봉인기에 각인된 드럼관리번호를 관리하는 기능 구현(아울러 QR코드 연계 URL 및 앱에서 등급판정 확인서 열람 등 소비자용 앱 조회기능도 고도화)

Q28: [X]


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


================= DEBUG Q29 =================
Q: 꿀 품질평가 사업 보안 지침상 '경미한 위반 3건' 이상 발생 시 어떤 등급의 위약금을 물게 되나요?
GOLD: D급(200만 원 이하 위약금)입니다.

--- PRED_CORE(300) ---
근거 문서에서 확인할 수 없습니다

Q29: [X]


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"


================= DEBUG Q30 =================
Q: 꿀 품질관리 시스템 누출금지 대상정보 중에서 2번 항목에 해당하는 구체적인 정보는 무엇인가요?
GOLD: 누출금지 대상정보 2번 항목입니다.(세부 정보시스템 구성현황 및 정보통신망 구성도)

--- PRED_CORE(300) ---
근거 문서에서 확인할 수 없습니다

Q30: [X]

평가 완료: 5 / 30


In [20]:
import pandas as pd

df = pd.DataFrame(results)
df_llm_judge = pd.DataFrame(results_llm_as_a_judge)
#display(df[["id","correct","score","llm_judge_score","question","gold","pred_core"]].head(10))

OUT = ROOT / "evaluation" / "eval_results_test_questions_2.csv"
df.to_csv(OUT, index=False, encoding="utf-8-sig")

OUT = ROOT / "evaluation" / "eval_results_test_questions_2_llm_judge.csv"
df_llm_judge.to_csv(OUT, index=False, encoding="utf-8-sig")

print("saved:", OUT)

saved: C:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project\evaluation\eval_results_test_questions_2_llm_judge.csv


In [22]:
df[["id","correct","score","llm_judge_score","question","gold","pred_core"]]

,id,correct,score,llm_judge_score,question,gold,pred_core
0,1,False,0.326087,None,봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 ...,"900,000,000원(VAT 포함)이며, 봉화군 안전건설과에서 주관합니다.","사업금액: 900,000,000원(부가세 포함). 주관부서: 근거 문서에서 확인할 ..."
1,2,True,0.520000,None,이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무...,LTE 및 DMB(EWS) 방식을 활용합니다.,LTE 및 DMB(EWS) 기반의 무선방식
2,3,False,0.250000,None,봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기...,100W 이상의 출력을 지원해야 합니다.,앰프출력 : 최대 400W 이상(100Wx4ch)
3,4,False,0.340909,None,고려대 차세대 학사시스템 구축 사업의 전체 예산 규모와 연도별 분할 지급 비율이 어...,"총 11,270,000,000원이며, 2024년 30%, 2025년 40%, 202...","전체 예산 11,270,000,000원(부가가치세 포함). 연도별 분할 지급 비율은..."
4,5,False,0.316456,None,고려대 포털 고도화 작업 중에 기존 'AI선배' 서비스의 아키텍처는 어떤 방향으로 ...,Python 기반 모델 아키텍처 고도화 및 서비스 통합 재설계입니다.,- 원데이터 추출·적재방식 및 데이터마트 생성방식을 경영통계시스템과 일원화 \n-...
5,6,True,0.365854,None,고려대 학사행정시스템을 구축할 때 서울과 세종 캠퍼스별 처리에 대한 별도의 지침이 ...,"표준화된 시스템 구축을 원칙으로 하되, 업무 성격상 캠퍼스 구분이 필요한 경우 이를...","네. 표준화된 통합 학사행정시스템을 구축하되, 조회/입력 등 기능 처리 시 서울캠퍼..."
6,7,False,0.125000,None,중앙의료원 응급의료 상황관리시스템 구축 사업에서 광역상황실이 신규로 설치되는 4개 ...,"서울, 대전, 대구, 광주 지역입니다.",근거 문서에서 확인할 수 없습니다
7,8,False,0.074074,None,국립중앙의료원 인프라 요건 중 DB 서버(Server-001)가 갖추어야 할 최소 ...,16 Cores 이상을 요구합니다.,근거 문서에서 확인할 수 없습니다
8,9,False,0.093750,None,응급환자 전원 지원 시스템의 핵심 업무 중 '상황의사'는 주로 어떤 역할을 수행하게...,"의뢰 대상 기관 선정, 수용 의뢰 및 상황 지원을 담당합니다.",근거 문서에서 확인할 수 없습니다
9,10,False,0.235294,None,나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역은 계약 후 기간이 총 얼마나...,총 18개월입니다.,계약체결일로부터 18개월


In [ ]:
df_llm_judge[["id","correct","score","llm_judge_score","question","gold","pred_core"]]

,id,correct,score,llm_judge_score,question,gold,pred_core
0,1,False,0.304348,None,봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 ...,"900,000,000원(VAT 포함)이며, 봉화군 안전건설과에서 주관합니다.","사업금액: 900,000,000원(부가세 포함)\n주관 부서: 근거 문서에서 확인할..."
1,2,False,0.317073,None,이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무...,LTE 및 DMB(EWS) 방식을 활용합니다.,"LTE 및 DMB(EWS) 기반 무선방식 (세부: LTE Cat 1, BAND 1/..."
2,3,False,0.173913,None,봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기...,100W 이상의 출력을 지원해야 합니다.,앰프출력: 최대 400W 이상(100W x 4ch). 또한 혼스피커는 8Ω/75W ...
3,4,True,0.432432,None,고려대 차세대 학사시스템 구축 사업의 전체 예산 규모와 연도별 분할 지급 비율이 어...,"총 11,270,000,000원이며, 2024년 30%, 2025년 40%, 202...","총 예산 11,270,000,000원(V.A.T 포함). 3년 분할지급: 2024학..."
4,5,False,0.333333,None,고려대 포털 고도화 작업 중에 기존 'AI선배' 서비스의 아키텍처는 어떤 방향으로 ...,Python 기반 모델 아키텍처 고도화 및 서비스 통합 재설계입니다.,기존 AI선배는\n- 경영통계시스템과 원데이터 추출·적재방식 및 데이터마트 생성방식...
5,6,False,0.319149,None,고려대 학사행정시스템을 구축할 때 서울과 세종 캠퍼스별 처리에 대한 별도의 지침이 ...,"표준화된 시스템 구축을 원칙으로 하되, 업무 성격상 캠퍼스 구분이 필요한 경우 이를...","예. 표준화된 통합 학사행정시스템으로 구현하되, 조회/입력 등 기능 처리 시 서울캠..."
6,7,False,0.125000,None,중앙의료원 응급의료 상황관리시스템 구축 사업에서 광역상황실이 신규로 설치되는 4개 ...,"서울, 대전, 대구, 광주 지역입니다.",근거 문서에서 확인할 수 없습니다
7,8,False,0.074074,None,국립중앙의료원 인프라 요건 중 DB 서버(Server-001)가 갖추어야 할 최소 ...,16 Cores 이상을 요구합니다.,근거 문서에서 확인할 수 없습니다
8,9,False,0.093750,None,응급환자 전원 지원 시스템의 핵심 업무 중 '상황의사'는 주로 어떤 역할을 수행하게...,"의뢰 대상 기관 선정, 수용 의뢰 및 상황 지원을 담당합니다.",근거 문서에서 확인할 수 없습니다
9,10,False,0.235294,None,나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역은 계약 후 기간이 총 얼마나...,총 18개월입니다.,계약체결일로부터 18개월
